# Lesson 4.2: Named Entity Recognition

**🎬 Video:** [Lesson 4.2: Named Entity Recognition](#)

## Overview

In this lesson, you will use spaCy's Named Entity Recognition (NER) to extract location names from Reddit posts. You will:

1. **Follow along** as NER identifies place names in individual sentences
2. **Apply NER** to all sentences in the dataset and build a toponym table
3. **Count and rank** the extracted place names by frequency
4. **Visualize** toponym frequencies with a bar chart
5. **Export** the processed data for geoparsing in Lesson 4.3

**Prerequisites:** [Lesson 4.1 — Extracting Toponyms in Texts](lesson_4_1_extracting_locations.ipynb)

---

## 📖 1 Follow Along — Using Named Entity Recognition

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Working with location names represents a unique computational problem. Since we do not know in advance what the names might be, and since some proper nouns can be both cities and people, (i.e. Jefferson, Washington, Lincoln, etc.) we need the computer to have some concept of what a place is. This is where language models come in. 

Language models are a form of **Machine Learning (ML)**. This is a branch of Artificial Intelligence (AI) in which a system learns patterns from large amounts of data rather than following hand-written rules. Machine learning models have been trained on a lot of text. Through statistical inference they establish the different types of **entities** or words in a text. These can be **parts of speech** like a verb, noun, adjective etc. With this basic understanding of grammar, they can infer more complex **entities** people, places, and organizations. 

One of the main libraries that Python uses to do this is `sPacy`. The sample below goes through the basic procedure for extracting an entity. 

**Don't worry about how the code works for now, just look at the result**

The code passes through the sentence:
>The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson.



In [ ]:
import spacy

# Load the small English model — "sm" = small, fast, good enough for demos
nlp = spacy.load("en_core_web_sm")

# Our example sentence — deliberately contains a person named Jefferson AND
# a place named Jefferson so we can see whether spaCy tells them apart
text = """The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson."""

# nlp() runs the full pipeline: tokenize → tag parts of speech → find entities
doc = nlp(text)

# This coding helps display the result in a human-readable format. Understanding how it works is not important.
print(f"Text:\n{text.strip()}\n")
print("Named entities:")
for ent in doc.ents:
    print(f"  {ent.text!r:30} {ent.label_!r:10} {spacy.explain(ent.label_)}")


> 📊 **Output:** Notice how accurately spaCy is able to distinguish between an organization in Virginia (UVA), a person Jefferson, and the place Jefferson (geopolitical entity).

spaCy is only as accurate as the data provided to it. If the text data is garbled or too short, it will likely have trouble. Undoubtedly, there are sentences in our `sentences` column that are not going to be read properly, but what we are relying on is the sheer volume of text. Even with some false positives and false negatives, we should be able to build a pretty good overview of the most mentioned places.

---

## 📖 2 Follow Along — Extract Entities in All Sentences

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The sentence-splitting work was done in lesson 4.1 and saved as a pickle. We load it here so we can run NER without repeating that step.

In [ ]:
import pandas as pd

df_reddit_sentences = pd.read_pickle("../data/JMU/JMU_sentences.pickle")
print(f"✅ Loaded {len(df_reddit_sentences):,} rows from JMU_sentences.pickle")

Doing one extraction on one sentence in `spacy` is pretty straight forward. We simply run the function `nlp()` on whatever sentence we want to analyze and save the result to a new variable, usually called `doc`. When we run this on a column with thousands of sentences, you start to run into performance issues because you are doing the procedure one at a time, and you also don't really know what's going on because there's no feedback. The functions below modify the above procedure a bit and basically asks your computer to use multiple processors. It also provides a little progress bar to make sure the computer is still working. Finally, instead of using the very small model that we used above, we are now going to use a slightly bigger model called `en_core_web_md`, this will hopefully help us find more locations!

We are now ready to apply this function to `sentences` and create a new column called `toponyms`. 

**Warning this process will take a couple of minutes**

In [ ]:
import spacy
from tqdm import tqdm

# Load the medium English model — more accurate than "sm" for location detection
nlp = spacy.load("en_core_web_md")

# nlp.pipe() processes all sentences in batches — much faster than one at a time.
# batch_size=256 means 256 sentences are grouped together per pass.
# tqdm displays a progress bar so you can track how far along the process is.
# fillna("") replaces any missing values with an empty string — spaCy requires strings.
sentences = df_reddit_sentences["sentences"].fillna("")
toponyms = []

for doc in tqdm(nlp.pipe(sentences, batch_size=256), total=len(sentences)):
    # For each sentence, collect any entities labelled GPE (Geopolitical Entity = places)
    gpes = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    toponyms.append(gpes if gpes else None)

# Store the results as a new column in the DataFrame
df_reddit_sentences["toponyms"] = toponyms

print(f"✅ Done — processed {len(sentences):,} sentences")


> 📊 **Output:** You should see around 16,000 sentences processed in about 1–3 minutes. Keep in mind that this is an incredibly complex task for a computer. For every single sentence it has to: read the text, parse the grammar, decide whether any words are locations, extract those locations, and store the result. It then repeats the process for the next sentence. The fact that it completes ~16,000 of these analyses in a few minutes is a remarkable feat of modern NLP. A human researcher doing this manually would need days if not weeks.

### Analyze the results

Run the code below to show the table and the results. The display has been separated from the processing because you do not want to process the data every time you want to view the results. 

In [ ]:
(
    df_reddit_sentences[["sentences", "toponyms"]]
    .dropna(subset=["toponyms"])
    .sample(5, random_state=92)
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)


> 📊 **Output:** Each row shows a sentence and the list of toponyms spaCy extracted from it. Some sentences have multiple place names in the list, because multiple places were found in the sentence. Other sentences may added places that are not actually places. This is because spaCy is making an educated guess based on context. It does not actually understand language in the way humans do. This is a common feature of all Machine learning. Still, this sample confirms the NLP pipeline ran successfully.

> 💡 **Reflection:** Look at the locations `spaCy` found in this sample. Which ones could you have found with a simple keyword list — city names you already knew to search for? Which ones might you never have thought to include? And are there any sentences that clearly mention a place that `spaCy` missed entirely? Keep those examples in mind: they are the evidence for why NER is more powerful than a list, but also why it is not perfect.

Because not every sentence includes a toponym, sometimes the output is `None`. We want to eliminate these rows because they are not relevant. Still, we might want to peek inside and calculate what percentage of sentences actually have toponyms. The calculation below achieves exactly this. It creates a table of the number of sentences with toponyms, and then divides the number of rows in that table by the total number of rows in the data set. This gives the percentage of rows that contain locations. In this case, around 4%.

In [ ]:
# Filter to sentences that have at least one toponym — None rows are dropped here
df_reddit_toponyms = df_reddit_sentences[df_reddit_sentences["toponyms"].notna()]
total = len(df_reddit_sentences)

print(f"Sentences with at least one toponym: {len(df_reddit_toponyms):,} of {total:,} ({len(df_reddit_toponyms) / total * 100:.1f}%)")

> 💡 **Reflection:** Only about 4% of sentences contain a place name. This is a pretty low number. What does that tell you about how people write on Reddit? Are most posts about events and opinions rather than places? If we are only using a small subset of sentences, how might that distort our results when we look at sentiment by locations?


---

## 📖 3 Follow Along — Counting Toponyms

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The code below counts how often each place name appears across the new dataset of place names. It does this in two steps:

1. **Flatten** — each row currently holds a *list* of toponyms (a sentence with two places has two items in its list). `.explode()` unpacks those lists so each toponym gets its own row, the same operation we used in Section 2.
2. **Count** — `.value_counts()` counts how many times each toponym appears and sorts the result from most to least common.

In [ ]:
# Step 1: Flatten the lists — each toponym gets its own row (same .explode() we used earlier)
unnested = df_reddit_toponyms["toponyms"].explode()

# Step 2: Count and sort — .value_counts() counts occurrences, most common first
toponym_counts_df = (
    unnested
    .value_counts()
    .reset_index()
    .rename(columns={"toponyms": "Toponym", "count": "Count"})
)

toponym_counts_df.head(10)

> 💡 **Reflection:** Not surprisingly, Harrisonburg is the top toponym. Scan the rest of the list and see if there are any entries that are not actually places or if something has been misidentified as two different places. 

---

## 4 Visualizing Toponyms



We can follow the same procedure from [lesson 3.2](../lesson_3_introduction_pandas/lesson_3_2_plotly_charts.ipynb) and put the places on a bar chart by count. `px.bar()` takes the DataFrame we just built and maps `Toponym` to the x-axis and `Count` to the y-axis.

In [ ]:
import plotly.express as px

# Take the top 10 most common toponyms for plotting
toponym_counts_top10 = toponym_counts_df.head(10)

# Create the bar chart using Plotly
fig = px.bar(
    toponym_counts_top10,
    x="Toponym",
    y="Count",
    title="Top 10 Most Common Toponyms",
    text="Count"
)

# Display the plot
fig.show()

> 💡 **Reflection:** Some of the names on the list are synonyms. If you were to stack these bars up on top of each other, what would the chart look like? What three places dominate discussion on the reddit thread?

### 3.2 Engagement by Toponym

Raw counts tell us what places are *mentioned* most often. But this figure can be misleading. For example, one entry contains the address for the President's office at JMU. 

>"James Madison University · 91 Alumnae Dr MSC 7608 · Harrisonburg, VA 22807 · president@jmu.edu"

This appears to be an entirely informational post with little engagement. By dint of being about JMU, many of the entries might mention Harrisonburg, but something else might be driving engagement. 

To understand what locations are driving engagement we need to know two things:

1. How often a location occurred
2. The average score of all the posts 

The code below computes both in a single step using `.groupby().agg()`:

| Step | What it does |
|---|---|
| **`.explode()`** | Same as before — one toponym per row |
| **`.groupby('toponyms')`** | Group all rows that share the same place name |
| **`.agg(...)`** | For each group, compute multiple summary statistics at once — here, `count` and `mean` |

In [ ]:
# Flatten: one toponym per row, keep the post score alongside it
toponym_score_df = (
    df_reddit_toponyms[["toponyms", "score"]]
    .explode("toponyms")
    .dropna(subset=["toponyms"])
)

# Group by toponym: count mentions AND average the post score
# Filter to places mentioned at least 10 times — rare mentions can have
# artificially high average scores, so a minimum count gives cleaner results
toponym_engagement = (
    toponym_score_df
    .groupby("toponyms", as_index=False)
    .agg(Count=("toponyms", "count"), Avg_Score=("score", "mean"))
    .rename(columns={"toponyms": "Toponym"})
    .query("Count >= 10")
    .sort_values(["Avg_Score", "Count"], ascending=False)
    .reset_index(drop=True)
)

toponym_engagement.head(15)

> 📊 **Output:** The table shows toponyms ranked by engagement. `Count` is the number of times a place was mentioned; `Avg_Score` is the mean upvote score of posts that mentioned it. Places at the top were both frequently mentioned and generated high engagement.

Visualizing this data can be quite tricky because there are two numbers that mean quite different things. One is a frequency count *how often* and the other is an intensity metric, *how did users respond*. What's more, the numbers are on different scales. Location mentions range from 1-110 while the highest average score is 39. It is not possible to put these on the same axis. Instead, the chart below is a tree map that shows the relative size of each toponym by the area of the square and the intensity of engagement through the color of the square.

In [ ]:
# Take the top 30 by engagement for plotting
top30_engagement = toponym_engagement.head(30)

# Treemap: each box is one place name
#   Box SIZE  → Count        (bigger box = mentioned more often)
#   Box COLOR → Avg_Score    (darker blue = higher average upvote score)
fig = px.treemap(
    top30_engagement,
    path=["Toponym"],
    values="Count",
    color="Avg_Score",
    title="Toponyms: Box Size = Mentions · Color = Average Upvote Score",
    labels={"Avg_Score": "Avg Score", "Count": "Mentions"},
    color_continuous_scale="Blues",
)

fig.update_traces(textinfo="label+value")
fig.show()

> 💡 **Reflection:** In this chart, box size shows how often a place was mentioned and color shows how much engagement those posts generated. Which places are both large *and* dark? That is, what places were both frequently mentioned and highly engaging? Are there any places with a small box but a deep color, meaning they come up rarely but generate a strong reaction when they do? What might explain that pattern?

---

## 📖 5 Follow Along — Save & Export

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

In [ ]:
# Save the filtered DataFrame (sentences that have toponyms) as a pickle.
# This preserves dtypes so the next notebook doesn't have to re-cast everything.
df_reddit_toponyms.to_pickle("../data/JMU/JMU_toponyms.pickle")
print("✅ Saved ../data/JMU/JMU_toponyms.pickle")

---

## Lesson Summary

### Part 1: Using Named Entity Recognition
- `spacy.load("model_name")` — loads a pre-trained NLP model into a variable (`nlp`) ready to process text
- `spacy.explain("LABEL")` — returns a human-readable description of any spaCy entity or part-of-speech label

### Part 2: Extract Entities in All Sentences
- `nlp.pipe(texts)` — runs the NLP model efficiently over a large list of strings in batches
- `ent.label_` — the entity type assigned by the model (e.g. `GPE` for geopolitical entities, `LOC` for locations)
- `ent.text` — the raw string the model identified as a named entity

### Part 3: Counting Toponyms
- `df['col'].value_counts()` — counts how often each unique value appears; useful for ranking place-name frequency

### Part 4: Visualizing Toponyms
- `px.bar(df, x=..., y=...)` — creates a bar chart from a DataFrame column; used here to rank toponyms by frequency and engagement

### Part 5: Save & Export
- `df.to_pickle('file.pickle')` — saves the DataFrame to disk so the next notebook can load it without repeating the NER step

➡️ **Next:** [Lesson 4.3 — Geoparsing in Python](lesson_4_3_geoparsing_mapping.ipynb)
